**Import Libraries**

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from pathlib import Path

import lightgbm as lgb
import optuna
import joblib

from sklearn.model_selection import (
    KFold,
    train_test_split
)

from sklearn.feature_selection import (
    VarianceThreshold,
    RFE
)

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

**Load Dataset**

In [ ]:
DATA = Path("../data/processed")

df = pd.read_csv(
    DATA/"final_features.csv"
)

print(df.shape)

**Build X and Y**

In [ ]:
TARGET = "VO2max"

X = df.drop(columns=[TARGET])

y = df[TARGET]

**Train / Test Split**

In [ ]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

**Remove Near-Zero Variance Features**

In [ ]:
selector = VarianceThreshold(
    threshold=0.01
)

X_dev = selector.fit_transform(X_dev)

X_test = selector.transform(X_test)

selected_columns = X.columns[
    selector.get_support()
]

print(selected_columns)

**Recursive Feature Elimination**

In [ ]:
base_model = lgb.LGBMRegressor(
    objective="regression",
    random_state=42
)
rfe = RFE(
    estimator=base_model,
    n_features_to_select=12
)

X_dev = rfe.fit_transform(
    X_dev,
    y_dev
)

X_test = rfe.transform(
    X_test
)

selected_columns = selected_columns[
    rfe.get_support()
]
outer_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
outer_scores = []
best_params_list = []

**Optuna Objective**

In [ ]:
def objective(trial):

    params = {

        "objective":"quantile",

        "alpha":0.5,

        "metric":"quantile",

        "verbosity":-1,

        "learning_rate":
            trial.suggest_float(
                "learning_rate",
                0.005,
                0.2,
                log=True
            ),

        "num_leaves":
            trial.suggest_int(
                "num_leaves",
                20,
                200
            ),

        "max_depth":
            trial.suggest_int(
                "max_depth",
                3,
                12
            ),

        "feature_fraction":
            trial.suggest_float(
                "feature_fraction",
                0.6,
                1.0
            ),

        "bagging_fraction":
            trial.suggest_float(
                "bagging_fraction",
                0.6,
                1.0
            ),

        "min_child_samples":
            trial.suggest_int(
                "min_child_samples",
                5,
                80
            ),

        "lambda_l1":
            trial.suggest_float(
                "lambda_l1",
                0,
                10
            ),

        "lambda_l2":
            trial.suggest_float(
                "lambda_l2",
                0,
                10
            ),

        "n_estimators":
            trial.suggest_int(
                "n_estimators",
                300,
                3000
            )
    }

    inner_scores=[]

    inner_cv = KFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    for train_idx,val_idx in inner_cv.split(X_train):

        X_tr=X_train[train_idx]
        X_val=X_train[val_idx]

        y_tr=y_train.iloc[train_idx]
        y_val=y_train.iloc[val_idx]

        model=lgb.LGBMRegressor(
            **params
        )

        model.fit(

            X_tr,

            y_tr,

            eval_set=[(X_val,y_val)],

            callbacks=[
                lgb.early_stopping(
                    100,
                    verbose=False
                )
            ]
        )

        pred=model.predict(X_val)

        rmse=np.sqrt(
            mean_squared_error(
                y_val,
                pred
            )
        )

        inner_scores.append(
            rmse
        )

    return np.mean(inner_scores)

**Nested Cross Validation**

In [ ]:
for fold,(train_idx,val_idx) in enumerate(
        outer_cv.split(X_dev)
):

    print(f"Outer Fold {fold+1}")

    X_train = X_dev[train_idx]
    X_valid = X_dev[val_idx]

    y_train = y_dev.iloc[train_idx]
    y_valid = y_dev.iloc[val_idx]

    study = optuna.create_study(
        direction="minimize"
    )

    study.optimize(
        objective,
        n_trials=50
    )

    best_params = study.best_params

    best_params["objective"]="quantile"

    best_params["alpha"]=0.5

    model = lgb.LGBMRegressor(
        **best_params
    )

    model.fit(

        X_train,

        y_train,

        eval_set=[(X_valid,y_valid)],

        callbacks=[
            lgb.early_stopping(
                100
            )
        ]
    )

    pred=model.predict(
        X_valid
    )

    rmse=np.sqrt(
        mean_squared_error(
            y_valid,
            pred
        )
    )

    outer_scores.append(
        rmse
    )

    best_params_list.append(
        best_params
    )

**CV Results**

In [ ]:
print("RMSE per Fold")

print(outer_scores)

print()

print("Mean RMSE")

print(np.mean(outer_scores))

print()

print("Std RMSE")

print(np.std(outer_scores))

**Train Final Models**

In [ ]:
alphas = [0.25, 0.50, 0.75]
models = {}

for alpha in alphas:
    params = best_params.copy()
    params["objective"] = "quantile"
    params["alpha"] = alpha

    model = lgb.LGBMRegressor(**params)
    model.fit(X_dev, y_dev)

    models[alpha] = model

**Save Artifacts**

In [ ]:
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

for alpha, model in models.items():
    joblib.dump(
        model,
        MODEL_DIR / f"lightgbm_q{int(alpha*100)}.pkl"
    )

joblib.dump(
    selector,
    MODEL_DIR / "variance_selector.pkl"
)

joblib.dump(
    rfe,
    MODEL_DIR / "rfe_selector.pkl"
)

joblib.dump(
    list(selected_columns),
    MODEL_DIR / "selected_features.pkl"
)